# 07 — ML-6: Causal CUSUM, Blind Holdout ve Operasyonel Alarm

Bu notebook ML-1..5 sonuçlarının metodolojik denetimini kapatır. CUSUM parametreleri artık yalnız `split_00` normal train uçuşlarında öğrenilir ve `artifacts/cusum/` altında saklanır. UAV-SEAD anomalileri sabit development/blind-holdout oturumlarına ayrılır; bu notebook yalnız development sonuçlarını gösterir.

Deneyleri yeniden üretmek için:
```powershell
python -m src.ml.build_features
python -m scripts.run_ml6_remeasure
python -m scripts.run_ml6_ablation
python -m scripts.run_ml6_thresholds
python -m scripts.run_ml6_events
python -m scripts.package_ml6_iforest
python -m scripts.package_ml6_lstm
```

In [ ]:
import json
from pathlib import Path
import pandas as pd

ROOT = Path.cwd() if (Path.cwd() / 'data').exists() else Path.cwd().parent
ML6 = ROOT / 'data/gold/ml6'
manifest = json.loads((ROOT / 'data/gold/ml_features/split_manifest.json').read_text(encoding='utf-8'))
split = manifest['sources']['uav_sead']['splits']['split_00']
print({k: len(split[k]) for k in ['train','val','test','final_holdout']})
print('blind anomaly:', len(split['final_holdout_anomalous']))
print('split unit:', manifest['sources']['uav_sead']['split_unit'])

## 1. Causal CUSUM yeniden ölçümü

Eski uçuş-içi MAD kullanan `alt_error_cusum_pos` ROC=0.878 sonucu geçersizdir. Train-normal sabit baseline ile causal yeniden ölçümde ROC=0.611 olmuştur; en güçlü tek ALFA sinyali artık `xtrack_error` (0.751) olmuştur. LSTM-AE giriş listesinde CUSUM bulunmadığı için ALFA LSTM-AE sonucu bu düzeltmeden doğrudan etkilenmez.

In [ ]:
single = pd.read_csv(ML6 / 'causal_cusum_single_features.csv')
modular = pd.read_csv(ML6 / 'causal_modular_remeasure.csv')
display(single.round(3))
display(modular.groupby('source')[['flight_roc','detection_at_1','false_alarm_at_1']].agg(['mean','std']).round(3))

## 2. Veri büyütme × split 2×2 ablation

349 uçuşa büyütme ROC'u ve tespiti artırmıştır. Session split her iki veri boyutunda da seed varyansını azaltmıştır; 349/session koşulu ROC 0.678 ± 0.013 ile en kararlıdır. Blind final holdout bu tablolara dahil değildir.

In [ ]:
ablation = pd.read_csv(ML6 / 'sead_2x2_ablation.csv')
display(ablation.groupby(['n_flights','split_unit'])[['flight_roc','detection_at_1','false_alarm_at_1']].agg(['mean','std']).round(3))

## 3. Threshold ve event alarmı

EKF'siz varsayılan modelde POT yanlış alarmı azaltırken tespiti de düşürmüştür; bu nedenle tek başına varsayılan ilan edilmez. K-of-N persistence yanlış alarm episode'larını azaltmıştır fakat false-alarm/hour halen deploy için yüksektir. Blind holdout, eşik ve persistence politikası dondurulana kadar açılmaz.

In [ ]:
thresholds = pd.read_csv(ML6 / 'sead_base_thresholds.csv')
events = pd.read_csv(ML6 / 'sead_event_metrics.csv')
display(thresholds.groupby('threshold_method')[['detection','false_alarm']].agg(['mean','std']).round(3))
summary = []
for policy, g in events.groupby('policy'):
    summary.append({'policy': policy, 'event_recall': g.detected_events.sum()/g.n_events.sum(),
                    'false_alarms_per_hour': g.false_alarm_events.sum()/g.normal_hours.sum()})
display(pd.DataFrame(summary).round(3))